In [ ]:
# Shared paths: configure raw data once in config.local.toml at the repo root.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_project = next(
    (p for p in (_start, *_start.parents) if (p / "scripts" / "project_paths.py").is_file()),
    None,
)
if _project is None:
    raise RuntimeError("Start the notebook kernel in the repository or a subdirectory.")
_scripts = str(_project / "scripts")
if _scripts not in sys.path:
    sys.path.insert(0, _scripts)
from project_paths import PROJECT_ROOT, MIMIC_DATA_DIR, PROCESSED_DIR, REPORTS_DIR, mimic_csv


In [1]:
import pandas as pd
from pathlib import Path

processed_path = PROCESSED_DIR

train_data = pd.read_parquet(processed_path / "ml_train.parquet")
test_data = pd.read_parquet(processed_path / "ml_test.parquet")

X_train = train_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_train = train_data["HOSPITAL_EXPIRE_FLAG"]
subject_id_train = train_data["SUBJECT_ID"]

X_test = test_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_test = test_data["HOSPITAL_EXPIRE_FLAG"]

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

categorical_cols = [
    "gender",
    "admission_type",
    "admission_location"
]

numeric_cols = X_train.columns.difference(categorical_cols).tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_cols),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", xgb_model)
    ]
)

In [4]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

setup = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

xgb_results = cross_validate(
    xgb_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    scoring=["roc_auc", "precision", "recall", "f1"]
)

print("ROC-AUC:", xgb_results["test_roc_auc"].mean())
print("Precision:", xgb_results["test_precision"].mean())
print("Recall:", xgb_results["test_recall"].mean())
print("F1:", xgb_results["test_f1"].mean())

ROC-AUC: 0.8545441899158602
Precision: 0.6060422984442181
Recall: 0.3140794831061688
F1: 0.4134369958057009


In [5]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5],
    "model__learning_rate": [0.05, 0.1]
}

xgb_grid_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    cv=setup,
    scoring="roc_auc",
    n_jobs=1
)

xgb_grid_search.fit(
    X_train,
    y_train,
    groups=subject_id_train
)

print("Best parameters:", xgb_grid_search.best_params_)
print("Best ROC-AUC:", xgb_grid_search.best_score_)

Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 200}
Best ROC-AUC: 0.8702302287462244


In [6]:
best_xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

best_xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", best_xgb_model)
    ]
)

best_xgb_results = cross_validate(
    best_xgb_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    scoring=["roc_auc", "precision", "recall", "f1"]
)

print("ROC-AUC:", best_xgb_results["test_roc_auc"].mean())
print("Precision:", best_xgb_results["test_precision"].mean())
print("Recall:", best_xgb_results["test_recall"].mean())
print("F1:", best_xgb_results["test_f1"].mean())

ROC-AUC: 0.8702302287462244
Precision: 0.6656192289804634
Recall: 0.30044720928059376
F1: 0.41382762087162084


In [7]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Negative:", negative_count)
print("Positive:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

Negative: 31842
Positive: 4327
scale_pos_weight: 7.358909174948001


In [8]:
balanced_xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=7.3589,
    random_state=42,
    eval_metric="logloss"
)

balanced_xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", balanced_xgb_model)
    ]
)

balanced_xgb_results = cross_validate(
    balanced_xgb_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    scoring=["roc_auc", "precision", "recall", "f1"]
)

print("ROC-AUC:", balanced_xgb_results["test_roc_auc"].mean())
print("Precision:", balanced_xgb_results["test_precision"].mean())
print("Recall:", balanced_xgb_results["test_recall"].mean())
print("F1:", balanced_xgb_results["test_f1"].mean())

ROC-AUC: 0.8644547035230257
Precision: 0.3815920395098421
Recall: 0.6799174998998785
F1: 0.4887133517648158


In [9]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

xgb_probs = cross_val_predict(
    balanced_xgb_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    method="predict_proba"
)[:, 1]

thresholds = [0.5, 0.6, 0.7, 0.8]

for threshold in thresholds:
    y_pred = (xgb_probs >= threshold).astype(int)

    print("Threshold:", threshold)
    print("Precision:", precision_score(y_train, y_pred))
    print("Recall:", recall_score(y_train, y_pred))
    print("F1:", f1_score(y_train, y_pred))
    print()

Threshold: 0.5
Precision: 0.3813844957220638
Recall: 0.6799168014790848
F1: 0.4886637322481521

Threshold: 0.6
Precision: 0.44580848824973696
Recall: 0.587474000462214
F1: 0.5069299032804866

Threshold: 0.7
Precision: 0.5193998965338852
Recall: 0.46406286110469147
F1: 0.49017453924081533

Threshold: 0.8
Precision: 0.6119469026548673
Recall: 0.3196209845158308
F1: 0.41991802034310005



## XGBoost

An XGBoost classifier was evaluated for ICU mortality prediction.

- Numerical features were processed using median imputation.
- Categorical features were transformed using `OneHotEncoder`.
- StandardScaler was not used because XGBoost is not sensitive to feature scaling.
- Model performance was evaluated using 5-fold `StratifiedGroupKFold`, keeping ICU stays from the same patient in the same fold.

### Baseline XGBoost

The default XGBoost model produced:

- ROC-AUC: **0.855**
- Precision: **0.606**
- Recall: **0.314**
- F1: **0.413**

The model showed good discrimination and precision, but recall remained relatively low.

### Hyperparameter Tuning

`GridSearchCV` was used to test different XGBoost settings.

The best parameters were:

- `n_estimators = 200`
- `max_depth = 5`
- `learning_rate = 0.1`

With these parameters:

- ROC-AUC: **0.870**
- Precision: **0.666**
- Recall: **0.300**
- F1: **0.414**

Hyperparameter tuning improved ROC-AUC and precision, but recall remained low.

### Class Imbalance Adjustment

Because mortality is the minority class, `scale_pos_weight` was used to give more importance to positive cases.

The class ratio was calculated as:

- Negative class: **31,842**
- Positive class: **4,327**
- `scale_pos_weight ≈ 7.36`

With class balancing:

- ROC-AUC: **0.864**
- Precision: **0.382**
- Recall: **0.680**
- F1: **0.489**

Class balancing substantially increased recall, while precision decreased as expected.

### Threshold Tuning

Different probability thresholds were evaluated using out-of-fold predictions from the tuned and class-balanced XGBoost model.

- Threshold `0.5`: Precision **0.381**, Recall **0.680**, F1 **0.489**
- Threshold `0.6`: Precision **0.446**, Recall **0.587**, F1 **0.507**
- Threshold `0.7`: Precision **0.519**, Recall **0.464**, F1 **0.490**
- Threshold `0.8`: Precision **0.612**, Recall **0.320**, F1 **0.420**

Threshold `0.6` provided the strongest overall precision-recall balance and the highest F1 score.

Overall, the best XGBoost configuration was:

- Tuned hyperparameters
- `scale_pos_weight ≈ 7.36`
- Classification threshold = **0.6**

This configuration achieved a strong balance between precision and recall while maintaining high ROC-AUC.